# General Pretraining — Consolidated Evaluation & Research Record

**One-file, read-only Jupyter Notebook** combining saved results from the two supplied evaluation notebooks and the previously completed duplicate/unique-molecule audit. No external `.py` helper, no Slurm job, no model inference, no rebuilding benchmark conformers, no raw-LMDB scan, and **no repeat of the completed unique-molecule analysis**.

**Order:** (1) locate existing HPC outputs → (2) pretraining reconstruction/geometry → (3) train–validation diagnostics → (4) representation PCA → (5) downstream property benchmarks → (6) duplicate/unique identity + scaffold/3D evidence → (7) research conclusions.

Run **Kernel → Restart & Run All**. The notebook reads already-saved figures and CSV/JSON results. Missing outputs are reported as missing, **never silently recomputed**. All images remain on HPC; this Notebook displays them inline. The original checkpoint/source datasets are not modified.

## 0. Configure existing HPC result directories (read-only)

In [ ]:
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Change these ONLY if HPC mounted a different location.
GENERAL_CANDIDATES = [
    '/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/mnt/project/yli4/ypan/heart_evaluation/general_audit_10k_v2',
    '/home/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2',
    '/scratch/ypan/heart_liver/model_train/heart_evaluation/general_audit_10k_v2',
]
DUPLICATE_CANDIDATES = [
    '/home/ypan/heart_liver/model_train/heart_evaluation/duplicate_impact_expanded_evidence',
    '/scratch/ypan/heart_liver/model_train/heart_evaluation/duplicate_impact_expanded_evidence',
    '/project/yli4/ypan/heart_evaluation/duplicate_impact_expanded_evidence',
    '/mnt/project/yli4/ypan/heart_evaluation/duplicate_impact_expanded_evidence',
]
def first_existing(paths):
    return next((Path(x) for x in paths if Path(x).is_dir()), None)
GENERAL = first_existing(GENERAL_CANDIDATES)
DUPLICATE = first_existing(DUPLICATE_CANDIDATES)
# Manually override GENERAL/DUPLICATE above if the auto-discovery did not find them.
print('General evaluation output:', GENERAL or 'NOT FOUND: set GENERAL = Path(...)')
print('Unique/dedup audit output:', DUPLICATE or 'NOT FOUND: set DUPLICATE = Path(...)')

def show_image(path, width=1050):
    p=Path(path)
    if p.is_file():
        display(Markdown(f'**Saved figure:** `{p.name}`'))
        display(Image(filename=str(p),width=width))
        return True
    return False

def find_images(root, include=(), exclude=()):
    if root is None: return []
    inc=[str(s).lower() for s in include]
    exc=[str(s).lower() for s in exclude]
    return sorted((p for p in root.rglob('*.png') if
                   all(s in p.name.lower() for s in inc) and
                   not any(s in p.name.lower() for s in exc)),key=lambda p:(len(p.parts),p.name))

def show_matches(root, include=(), exclude=(), max_figs=3):
    matches=find_images(root,include,exclude)
    if not matches:
        print('Saved figure not found:', ' / '.join(include), '\n  Root:',root)
        return []
    for p in matches[:max_figs]:show_image(p)
    if len(matches)>max_figs:print('Additional matching figures:',len(matches)-max_figs)
    return matches

def find_file(root,filename):
    if root is None:return None
    found=sorted(root.rglob(filename),key=lambda p:len(p.parts))
    return found[0] if found else None

def read_csv_or_none(p):
    return pd.read_csv(p) if p is not None and Path(p).is_file() else None

def section(msg):display(Markdown('**Interpretation:** '+msg))

print('Kernel:',sys.executable)
print('All following cells are cached-results-only; no GPU required.')

## 1. Saved 10K evidence — masked atom reconstruction and 3D denoising

**Question:** Did the completed general checkpoint learn the objectives used during pretraining?

The masked atom metric is **accuracy (higher is better)**, whereas coordinate RMSE is **error (lower is better)**. The 75.6% quantity below is **RMSE reduction**, not percent of 3D conformers recovered.

In [ ]:
if GENERAL:
    RUNS=GENERAL/'checkpoint_evaluation'
    valid_json=RUNS/'full_10k_batch8'/'evaluation_summary.json'
    train_json=RUNS/'train_10k_batch8'/'evaluation_summary.json'
    val=json.loads(valid_json.read_text()) if valid_json.is_file() else {}
    tr=json.loads(train_json.read_text()) if train_json.is_file() else {}
    names={'n_molecules':'Molecules','masked_atom_accuracy':'Masked atom accuracy',
           'masked_atom_majority_class_baseline_accuracy':'Majority baseline',
           'coord_noisy_rmse':'Noisy coordinate RMSE (Å)',
           'coord_pred_rmse':'Recovered coordinate RMSE (Å)',
           'coord_rmse_reduction_fraction':'RMSE reduction fraction',
           'contrastive_top1_both_directions':'Batch top-1 (batch 8)',
           'positive_projection_cosine_mean':'Positive pair cosine'}
    if val:
        display(pd.DataFrame([{'Metric':label,'Train':tr.get(k,np.nan),'Validation':val.get(k,np.nan)}
                              for k,label in names.items()]).round(5))
        if val.get('coord_pred_rmse') is not None and val.get('coord_noisy_rmse'):
            print('Coordinate RMSE reduction: {:.1%}'.format(1-val['coord_pred_rmse']/val['coord_noisy_rmse']))
    else:print('Existing 10K evaluation_summary.json not found:',valid_json)
    pics=find_images(GENERAL,('structure','geometry'))
    if not pics:pics=find_images(GENERAL,('slide_1',))
    if pics:show_image(pics[0])
    elif val:
        fig,axes=plt.subplots(1,2,figsize=(11,3.8))
        axes[0].bar(['Model','Majority baseline'],[val['masked_atom_accuracy']*100,val['masked_atom_majority_class_baseline_accuracy']*100]);axes[0].set(ylabel='Masked atom accuracy (%)',title='Validation 10K')
        axes[1].bar(['Noisy input','Recovered'],[val['coord_noisy_rmse'],val['coord_pred_rmse']]);axes[1].set(ylabel='Coordinate RMSE (Å) — lower is better',title='3D denoising')
        plt.tight_layout();plt.show()
else:print('Set GENERAL above to load cached 10K results.')
section('Strong reconstruction and coordinate denoising on the original objective; it does not establish RNA-delivery prediction or de-novo conformer generation.')

## 2. Per-element masked accuracy — identifying rare-element limitations

In [ ]:
if GENERAL:
    atom_csv=RUNS/'full_10k_batch8'/'atom_token_class_accuracy.csv'
    atom=read_csv_or_none(atom_csv)
    if atom is not None:
        display(atom)
        required={'token_symbol','accuracy','masked_targets'}
        if required <= set(atom.columns):
            x=atom[atom.masked_targets >= 20].copy()
            fig,ax=plt.subplots(figsize=(10,4.2))
            ax.bar(x.token_symbol,x.accuracy*100)
            ax.set(title='Per-element masked atom recovery — validation 10K',ylabel='Accuracy (%)',ylim=(0,105))
            for j,r in enumerate(x.itertuples()):ax.text(j, getattr(r,'accuracy')*100+1,str(getattr(r,'masked_targets')),ha='center',fontsize=8)
            plt.tight_layout();plt.show()
    else:print('Missing cached atom CSV:',atom_csv)
section('C/N/O/S/F perform strongly; Br and I have much fewer masked targets and less reliable accuracy estimates. Overall 98% accuracy is dominated by common atoms.')

## 3. Matched 10K training vs validation — checkpoint loss diagnostics

In [ ]:
if GENERAL:
    trcsv=RUNS/'train_10k_batch8'/'objective_batches.csv'
    vacsv=RUNS/'full_10k_batch8'/'objective_batches.csv'
    a,b=read_csv_or_none(trcsv),read_csv_or_none(vacsv)
    if a is not None and b is not None:
        keys=[k for k in ['masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','official_weighted_loss'] if k in a and k in b]
        df=pd.DataFrame({'Objective':keys,'Train mean':[a[k].mean() for k in keys],
                         'Validation mean':[b[k].mean() for k in keys]})
        df['Valid - train']=df['Validation mean']-df['Train mean']
        display(df.round(5))
        x=np.arange(len(df));fig,ax=plt.subplots(figsize=(11,4))
        ax.bar(x-.18,df['Train mean'],.36,label='Train 10K');ax.bar(x+.18,df['Validation mean'],.36,label='Validation 10K')
        ax.set_xticks(x,df['Objective'],rotation=20,ha='right');ax.set(ylabel='Original objective loss',title='Frozen checkpoint evaluation — matched sample sizes')
        ax.legend();plt.tight_layout();plt.show()
    else:print('Matched cached train/validation CSV unavailable; no evaluation will be run.')
section('Small train–validation loss gaps in sampled checkpoint evaluation. These are NOT optimizer-step training curves and cannot rule out all overfitting.')

## 4. Frozen molecular representations — saved PCA/UMAP

**Raw CLS:** PCA may reflect molecule size and other correlated structural properties. **NT-Xent projector:** lower PC variance and a diffuse PCA scatter do not by themselves mean poorer embeddings.

In [ ]:
if GENERAL:
    pcas=[p for p in GENERAL.rglob('*.png') if 'pca' in p.name.lower() and ('cls' in p.name.lower() or 'projector' in p.name.lower())]
    if pcas:
        for p in sorted(pcas)[:4]:show_image(p)
    else:print('No cached PCA PNG found. Representation arrays are not re-processed.')
section('PCA visualization supports association of raw CLS with molecular-size variation; it is exploratory, not a molecular-property predictive benchmark.')

## 5. Independent property benchmarks — frozen embeddings vs classical chemistry baselines

Saved **ESOL / FreeSolv / Lipophilicity** probe results are read from the original chemistry evidence directory. Ridge regression is fit on benchmark labels in the original analysis; **this notebook does not refit any probe**. Scaffold-disjoint downstream test splits do **not** guarantee no overlap with the original pretraining SMILES.

In [ ]:
if GENERAL:
    CHEM=GENERAL/'chemistry_slide_evidence'
    print('Cached chemistry output:', CHEM, '| exists:',CHEM.is_dir())
    if CHEM.is_dir():
        csvs=[p for p in CHEM.rglob('*.csv') if any(s in p.name.lower() for s in ['results','metrics','benchmark','summary','probe'])]
        print('Candidate benchmark tables:')
        for p in csvs[:30]:print(' ',p.relative_to(CHEM))
        for kw in ['esol','freesolv','lipophilicity']:
            print('\nBENCHMARK:',kw)
            plots=[p for p in CHEM.rglob('*.png') if kw in str(p).lower() and any(s in p.name.lower() for s in ['baseline','scaffold','efficiency','prediction','pred_vs','label'])]
            if not plots:plots=[p for p in CHEM.rglob('*.png') if kw in str(p).lower()]
            for p in sorted(plots,key=lambda z:(len(z.parts),z.name))[:3]:show_image(p)
    else:print('Chemistry evidence directory missing; set GENERAL to the original cached result tree.')
section('Frozen CLS is competitive on ESOL and Lipophilicity but not uniformly superior to RDKit/Morgan; FreeSolv is a counterexample where RDKit descriptors performed better.')

### 5a. Benchmark metric table (from saved output when possible)

Metrics for reference from previous evaluations (three scaffold splits, full labeled-data fraction, mean ± SD) are **shown only as a fallback** when their result CSV cannot be discovered. They are not newly calculated.

In [ ]:
if GENERAL and (GENERAL/'chemistry_slide_evidence').is_dir():
    cc=GENERAL/'chemistry_slide_evidence'
    candidates=[p for p in cc.rglob('*.csv') if p.name in ('benchmark_results.csv','results.csv','property_probe_results.csv','benchmark_summary.csv','probe_results.csv','external_benchmark_results.csv')]
    # Also accept an explicit result name discovered in older versions.
    candidates+= [p for p in cc.rglob('*.csv') if 'property' in p.name.lower() and 'result' in p.name.lower() and p not in candidates]
    shown=False
    for p in candidates:
        try:
            df=pd.read_csv(p)
            if {'benchmark','model','rmse'} <= set(df.columns):
                d=df.loc[df.train_fraction==1.0] if 'train_fraction' in df.columns else df
                display(Markdown(f'**Existing CSV:** `{p}`'))
                display(d.groupby(['benchmark','model'])['rmse'].agg(['mean','std','count']).round(3))
                shown=True;break
        except Exception as exc:print('Could not read',p.name,type(exc).__name__)
    if not shown:
        print('Cannot identify a compatible benchmark result CSV automatically. No model/probe rerun.')
        print('Previously recorded: ESOL HEART 0.910±0.031, RDKit 0.981±0.097, Morgan 1.713±0.207;')
        print('FreeSolv HEART 2.350±0.021, RDKit 1.828±0.244;')
        print('Lipophilicity HEART 0.924±0.058, Morgan 0.904±0.087, RDKit 1.015±0.054.')
else:print('Chemistry output not available.')

## 6. Saved structural-retrieval evidence and molecular geometry examples

Show only what already exists on HPC. Missing materials will be skipped without running nearest-neighbor similarity calculations or GPU model inference.

In [ ]:
if GENERAL and (GENERAL/'chemistry_slide_evidence').is_dir():
    cc=GENERAL/'chemistry_slide_evidence'
    for sub in ['structural_similarity','geometry_examples','slides_from_existing_10k']:
        d=cc/sub
        if d.is_dir():
            print('\nCached',sub)
            figs=sorted(d.rglob('*.png'))
            for p in figs[:4]:show_image(p)
            if not figs:print('No cached PNG.')
        else:print('Unavailable (skipped):',sub)

## 7. Existing full-dataset duplicate / unique identity audit — **never rerun**

This section reads `expanded_report.json`, CSV summaries and PNGs created by the *previously completed* unique-molecule/duplicate study. It does **not** open the 18.84M-record LMDB, regenerate hashes, bootstrap, sample scaffolds, or compare 3D conformers again.

In [ ]:
if DUPLICATE:
    print('Saved audit directory:',DUPLICATE)
    report=DUPLICATE/'expanded_report.json'
    if report.is_file():
        data=json.loads(report.read_text())
        print('Expanded report top-level fields:',list(data))
        for key,value in data.items():
            if isinstance(value,(int,float,str,bool)):
                print(f'{key}: {value}')
    else:print('Report JSON missing; existing saved figures may still be available.')
    for name in ['frequency_sample_summary.csv','scaffold_diversity.csv','duplicate_record_3d_pair_check.csv']:
        p=DUPLICATE/name
        if p.is_file():
            print('\nCached:',name)
            display(pd.read_csv(p).head(15))
    print('\nSaved count/exposure and slide overview figures:')
    for stem in ['00_PROGRESS_SLIDE_duplicate_impact','01_raw_vs_dedup_copy_frequency']:
        p=DUPLICATE/(stem+'.png')
        if p.is_file():show_image(p)
else:print('Duplicate evidence directory not found: set DUPLICATE in Section 0 to existing HPC path.')
section('18.84M raw records versus 13.27M unique raw SMILES (29.55% duplicate excess). Removing repeats changes exposure frequency; the evaluation below tests whether it materially changes measured chemical-space characteristics.')

## 8. Physicochemical distribution after deduplication — saved comparisons

In [ ]:
if DUPLICATE:
    for stem in ['03_true_raw_vs_dedup_descriptor_distribution','04_descriptor_effect_sizes_bootstrap']:
        p=DUPLICATE/(stem+'.png')
        if p.is_file():show_image(p)
        else:print('Figure missing:',p.name)
    for name in ['descriptor_direct_comparison_bootstrap.csv','chemical_descriptor_reweight_preview.csv']:
        p=DUPLICATE/name
        if p.is_file():
            print('\nSaved measurements:',name)
            display(pd.read_csv(p).head(20))
section('Raw and unique-molecule samples have highly overlapping measured distributions (e.g. TPSA KS≈0.010; rotatable bonds KS≈0.006). Small descriptive shifts are not a formal statistical equivalence result.')

## 9. Molecular scaffolds — saved equal-size sample comparison

In [ ]:
if DUPLICATE:
    p=DUPLICATE/'05_scaffold_diversity.png'
    if p.is_file():show_image(p)
    else:print('Missing saved scaffold figure:',p.name)
section('In equal-size n=3,000 samples, distinct Murcko scaffolds averaged about 2,731 (raw) versus 2,716 (dedup); effective diversity about 2,398 versus 2,360. No increase in sampled scaffold diversity was observed after deduplication.')

## 10. Stored duplicate 3D geometries — saved pairwise analysis

In [ ]:
if DUPLICATE:
    p=DUPLICATE/'06_3d_geometry_between_duplicate_records.png'
    if p.is_file():show_image(p)
    else:print('Missing saved geometry figure:',p.name)
    p=DUPLICATE/'duplicate_record_3d_pair_check.csv'
    if p.is_file():display(pd.read_csv(p).head(24))
section('24/24 sampled duplicate-record pairs had at least one near-identical coordinate slot at the exploratory 0.02 Å threshold. This does not prove all conformers across the pair are duplicated, nor generalize to all records.')

## 11. Interpretation, limitations, and next decision

**Supported by the existing saved evidence**

1. The completed general checkpoint shows strong masked-atom and coordinate-denoising performance on matched 10K validation samples.
2. Frozen CLS embeddings capture molecular variation and transfer to several external physicochemical property-label tasks, but do not uniformly outperform traditional RDKit/Morgan features.
3. The source contains considerable record-level SMILES duplication, yet the measured bulk descriptor distributions and equal-size scaffold-diversity comparisons change only modestly after deduplication.
4. The small sampled same-SMILES record-pair analysis provides preliminary evidence of duplicated 3D geometry.

**Important limits**

- These chemical-property benchmarks use scaffold-disjoint **labeled** splits, but some molecules have exact raw-SMILES overlap with the general pretraining corpus.
- Pretraining objective performance is **not** RNA/LNP delivery efficacy prediction.
- PCA, near-identical conformer pairs and bootstrap plots do **not** establish formal distributional equivalence or a downstream advantage from deduplicated pretraining.
- Current evidence supports **retaining the existing general checkpoint** and proceeding with the lipid-specific adaptation work; only a compute-matched ablation could establish performance effects of deduplication.

**Reproducibility policy:** All results above are loaded from saved HPC outputs. This notebook does not create raw-vs-unique comparisons, train probe models, run inference, or change data files.

## 12. Cached-file audit — what was located

In [ ]:
if GENERAL:
    print('GENERAL ROOT:',GENERAL)
    for sub in ['checkpoint_evaluation/full_10k_batch8/evaluation_summary.json',
                'checkpoint_evaluation/train_10k_batch8/evaluation_summary.json',
                'chemistry_slide_evidence/independent_property_benchmarks']:
        p=GENERAL/sub
        print('OK     ' if p.exists() else 'MISSING',p)
if DUPLICATE:
    print('DUPLICATE ROOT:',DUPLICATE)
    for name in ['expanded_report.json','00_PROGRESS_SLIDE_duplicate_impact.png',
                 '03_true_raw_vs_dedup_descriptor_distribution.png',
                 '05_scaffold_diversity.png','06_3d_geometry_between_duplicate_records.png']:
        p=DUPLICATE/name
        print('OK     ' if p.is_file() else 'MISSING',p)
print('Read-only summary complete. Nothing was rerun or written to the HPC source directories.')